# Skin-Only SyNRA Registration → Applied to Full Head_V6
> Compute the SyNRA warp from the **skin surface only**, then apply that one warp to
> every node in `Head_V6.k` (skull + brain + meninges + skin together) — the
> "unified transformation" principle from `architecture.md`: one warp, computed once
> from the external head shape, applied to every anatomical component so they deform
> together and stay aligned.

See `plan.md` (project root) and the vault note `Services/Skin-Only SyNRA Registration.md`
for the reasoning behind this vs. the whole-mesh approach in
`VTU_to_VTU_SyNRA_Registration.ipynb`.

| Cell | Stage |
|------|-------|
| 1 | Install |
| 2 | Config |
| 3 | Build skin-only MOVING mesh (merge 8 skin-part VTUs) |
| 4 | Load FIXED target + voxelize both (tuned for thin shell) |
| 5 | ANTsPy SyNRA registration (skin vs skin) |
| 5b | Metric 1 — image-domain Dice/IoU |
| 6 | Verify point-transform direction empirically (correctness gate, not optional) |
| 7 | Apply verified transform to skin nodes → registered skin `.vtu` (visual QA) |
| 8 | Parse full `Head_V6.k`, apply the *same* transform to every node |
| 9 | Write `Head_V6_registered.k` (round-trip node replace, all other cards preserved) |
| 10 | Metric 2 — `k_mesh_qa.py --compare` solver-safety gate on the full `.k` output |


## Cell 1 — Install

In [1]:
!pip install antspyx pyvista numpy matplotlib SimpleITK pandas -q

import ants
import pyvista as pv
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import pandas as pd
import os, time, subprocess, sys

print(f'ants    : {ants.__version__}')
print(f'pyvista : {pv.__version__}')
print(f'numpy   : {np.__version__}')


ants    : 0.6.3
pyvista : 0.48.4
numpy   : 2.2.5


## Cell 2 — Config

In [ ]:
# USER EDIT — adjust if your project root differs
PROJECT_ROOT   = '/home/asusry7/Desktop/New_version/New_version'
VTK_EXPORT_DIR = os.path.join(PROJECT_ROOT, 'Data', 'outputs', 'vtk_export')
PARTS_DIR      = os.path.join(VTK_EXPORT_DIR, 'parts')

FIXED_VTU      = os.path.join(VTK_EXPORT_DIR, 'ImageToStl.com_head_voxels_transform_2.vtu')
HEAD_V6_K      = os.path.join(PROJECT_ROOT, 'Head_V6.k')
K_MESH_QA_PY   = os.path.join(PROJECT_ROOT, 'Code', 'k_mesh_qa.py')

OUTPUT_DIR     = 'outputs/skin_only_synra'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Head_V6's full skin envelope = 24 parts from Head_V6_head_skin.k (face + head/scalp
# + nose, both hemispheres) — per the ground-truth PID table extracted directly from
# that .k file (2026-07-09). Previously this list had only 8 entries and, because of
# a title/PID off-by-one bug in K_to_VTK_Converter.ipynb (now fixed — see that
# notebook's Cell 3 and the vault note Services/K to VTK Converter.md), those 8 files
# were mislabeled 'head_skin_*' while actually containing 'face_skin_*' geometry —
# i.e. the moving mesh was face-only, not full-head, which is why registration
# against the full-head FIXED_VTU failed (Dice 0.297). K_to_VTK_Converter.ipynb has
# been re-run with the fix and parts/*.vtu filenames now match the real PID→name
# mapping; the 16 additional PIDs below (head_skin_*, nose_in_shell_*, and the
# face_skin_*_c[rl] center-strip parts) were previously entirely missing.
SKIN_PART_FILES = [
    'part_88000167_face_skin_right.vtu',
    'part_88000168_head_skin_right.vtu',
    'part_88000169_face_skin_cr.vtu',
    'part_88000170_face_skin_left.vtu',
    'part_88000171_head_skin_left.vtu',
    'part_88000172_face_skin_cl.vtu',
    'part_88000219_nose_in_shell_right.vtu',
    'part_88000220_nose_in_shell_left.vtu',
    'part_88000221_face_skin_out_r.vtu',
    'part_88000222_head_skin_out_r.vtu',
    'part_88000223_face_skin_out_cr.vtu',
    'part_88000224_face_skin_in_r.vtu',
    'part_88000225_head_skin_in_r.vtu',
    'part_88000226_face_skin_in_cr.vtu',
    'part_88000227_face_skin2_in_r.vtu',
    'part_88000228_head_skin2_in_r.vtu',
    'part_88000229_face_skin_out_l.vtu',
    'part_88000230_head_skin_out_l.vtu',
    'part_88000231_face_skin_out_cl.vtu',
    'part_88000232_face_skin_in_l.vtu',
    'part_88000233_head_skin_in_l.vtu',
    'part_88000234_face_skin_in_cl.vtu',
    'part_88000235_face_skin2_in_l.vtu',
    'part_88000236_head_skin2_in_l.vtu',
]

VOXEL_SIZE_MM = 0.75    # finer than the whole-mesh notebook's 1.0mm — skin is a thin
                        # shell (outer/inner surfaces close together); a coarser voxel
                        # grid risks merging them into one blob under Gaussian blur.
SYN_TYPE      = 'SyNRA'

print(f'Fixed target      : {FIXED_VTU}   exists={os.path.exists(FIXED_VTU)}')
print(f'Head_V6.k (full)  : {HEAD_V6_K}   exists={os.path.exists(HEAD_V6_K)}')
print(f'k_mesh_qa.py       : {K_MESH_QA_PY}   exists={os.path.exists(K_MESH_QA_PY)}')
missing = [f for f in SKIN_PART_FILES if not os.path.exists(os.path.join(PARTS_DIR, f))]
if missing:
    raise FileNotFoundError(f'Missing skin part VTUs: {missing}')
print(f'All {len(SKIN_PART_FILES)} skin part VTUs found.')


## Cell 3 — Build Skin-Only MOVING Mesh

Merge the 8 skin-part VTUs into one mesh. This is the MOVING side of the
registration — no full-body/full-head volume involved, just the external skin
envelope, matched against a target that is itself a skin surface.

In [3]:
skin_meshes = []
print('Loading skin part VTUs...')
for fname in SKIN_PART_FILES:
    path = os.path.join(PARTS_DIR, fname)
    m = pv.read(path)
    print(f'  {fname:<45} points={m.n_points:>7,}  cells={m.n_cells:>7,}')
    skin_meshes.append(m)

skin_mesh = skin_meshes[0].merge(skin_meshes[1:])
print(f'\nMerged skin mesh: {skin_mesh.n_points:,} points, {skin_mesh.n_cells:,} cells')

nodes_skin = np.array(skin_mesh.points, dtype=np.float64)
print(f'Bounds X: [{nodes_skin[:,0].min():.3f}, {nodes_skin[:,0].max():.3f}]')
print(f'Bounds Y: [{nodes_skin[:,1].min():.3f}, {nodes_skin[:,1].max():.3f}]')
print(f'Bounds Z: [{nodes_skin[:,2].min():.3f}, {nodes_skin[:,2].max():.3f}]')


Loading skin part VTUs...
  part_88000167_head_skin_right.vtu             points=  1,004  cells=    487
  part_88000170_head_skin_left.vtu              points=  1,004  cells=    487
  part_88000221_head_skin_out_r.vtu             points=    502  cells=    487
  part_88000224_head_skin_in_r.vtu              points=    272  cells=    198
  part_88000227_head_skin2_in_r.vtu             points=    353  cells=    289
  part_88000229_head_skin_out_l.vtu             points=    502  cells=    488
  part_88000232_head_skin_in_l.vtu              points=    272  cells=    198
  part_88000235_head_skin2_in_l.vtu             points=    353  cells=    289

Merged skin mesh: 1,958 points, 2,923 cells
Bounds X: [-171.298, -56.772]
Bounds Y: [-76.594, 76.594]
Bounds Z: [519.142, 677.417]


## Cell 4 — Load Fixed Target + Voxelize Both

Splat points, dilate, then fill solid (was Gaussian-blur-only, no fill -- see `voxelize_to_ants` docstring for the 2026-07-09 root-cause writeup: raw intensity-scale mismatch between fixed/moving starved the meansquares optimizer, and thin unfilled shells made Dice near-zero-tolerance for sub-voxel misalignment). Matches the solid-fill approach in the friend's `VTU_TwoStage_Registration_v3_3.ipynb` `nodes_to_ants_volume()`. Density oversampling for the sparse skin mesh is unchanged.

In [ ]:
def load_vtu(path):
    mesh  = pv.read(path)
    nodes = np.array(mesh.points, dtype=np.float64)
    print(f'  Points : {len(nodes):,}')
    print(f'  Cells  : {mesh.n_cells:,}')
    print(f'  Bounds X: [{nodes[:,0].min():.3f}, {nodes[:,0].max():.3f}]')
    print(f'  Bounds Y: [{nodes[:,1].min():.3f}, {nodes[:,1].max():.3f}]')
    print(f'  Bounds Z: [{nodes[:,2].min():.3f}, {nodes[:,2].max():.3f}]')
    return mesh, nodes


def auto_rescale(nodes_moving, nodes_fixed):
    """Rescale moving bounding box to match fixed, PER AXIS. Handles mm/m/cm
    mismatches -- and, just as importantly, anisotropic ones.

    Previously this used a single scalar (median of the 3 per-axis ratios)
    applied uniformly to X/Y/Z. For this data that was actively wrong: per-axis
    ratios came out [2.19, 1.38, 2.19] -- X and Z really do need ~2.19x, but Y
    only needs 1.38x. Forcing 2.19x onto Y stretched the skin's Y-extent to
    ~335mm against a ~211mm target (a ~124mm/59% baked-in mismatch) before
    registration even started, which SyNRA then had no real chance of undoing.
    Scaling each axis by its own ratio removes that self-inflicted mismatch
    instead of asking the registration to fix a scaling bug.
    """
    ext_f = nodes_fixed.max(axis=0)  - nodes_fixed.min(axis=0)
    ext_m = nodes_moving.max(axis=0) - nodes_moving.min(axis=0)
    scale = ext_f / (ext_m + 1e-8)   # per-axis (3,) vector, NOT a single scalar
    print(f'  Fixed  extent  : {ext_f.round(2)}')
    print(f'  Moving extent  : {ext_m.round(2)}')
    print(f'  Scale (X,Y,Z)  : {scale.round(4)}')
    if np.all((0.8 < scale) & (scale < 1.2)):
        print('  Units match on all axes — no rescaling needed')
        return nodes_moving, np.ones(3)
    c = nodes_moving.mean(axis=0)
    print(f'  Rescaling moving per-axis by {scale.round(4)} to match fixed space')
    return (nodes_moving - c) * scale + c, scale


def voxelize_to_ants(nodes, voxel_size=1.0, margin_mm=10.0, dilate_iters=2, tmp_path=None):
    """Point cloud -> binary volume -> dilate -> fill solid -> ANTs image.

    Was Gaussian-blur-only (no fill). That produced two problems confirmed
    2026-07-09: (1) the fixed target (dense, 405k pts) and moving skin
    (sparse, splatted+blurred) ended up on wildly different intensity scales
    after blur (~0.24 peak vs ~0.01 peak) -- `ants.registration(..., syn_metric=
    'meansquares', ...)` runs on these RAW un-normalized volumes, so the ~24x
    intensity gap starved the optimizer of gradient signal on the moving side.
    (2) thin unfilled shells give Dice almost no tolerance for sub-voxel
    misalignment (near-zero overlap volume even when the surface fit is
    reasonable), unlike the friend's VTU_TwoStage_Registration_v3_3.ipynb,
    which fills its skull volumes solid and scores well.

    Filling solid does NOT change what drives the registration -- meansquares
    gradient is zero deep inside (1 vs 1) and zero deep outside (0 vs 0); only
    the surface boundary disagreement carries signal, same as before. It just
    puts fixed/moving on the same intensity scale and gives Dice a sane,
    comparable denominator. The 24-part skin mesh's outer shell, once dilated
    and filled, also naturally absorbs the inner/outer skin double-layer gap
    into one solid head-shaped volume -- comparable to the fixed target, which
    is itself solid (voxel-derived STL).
    """
    from scipy.ndimage import binary_dilation, binary_fill_holes
    mins = nodes.min(axis=0) - margin_mm
    maxs = nodes.max(axis=0) + margin_mm
    dims = np.ceil((maxs - mins) / voxel_size).astype(int)
    vol  = np.zeros(dims[::-1], dtype=np.float32)
    idx  = np.clip(np.floor((nodes - mins) / voxel_size).astype(int), 0, dims-1)
    vol[idx[:,2], idx[:,1], idx[:,0]] = 1.0

    vol_d = binary_dilation(vol > 0, iterations=dilate_iters)
    fz = np.zeros_like(vol_d); fy = np.zeros_like(vol_d); fx = np.zeros_like(vol_d)
    for i in range(vol_d.shape[0]): fz[i,:,:] = binary_fill_holes(vol_d[i,:,:])
    for i in range(vol_d.shape[1]): fy[:,i,:] = binary_fill_holes(vol_d[:,i,:])
    for i in range(vol_d.shape[2]): fx[:,:,i] = binary_fill_holes(vol_d[:,:,i])
    solid = ((fz.astype(int) + fy.astype(int) + fx.astype(int)) >= 2).astype(np.float32)

    sitk_img = sitk.GetImageFromArray(solid)
    sitk_img.SetSpacing([voxel_size]*3)
    sitk_img.SetOrigin(mins.tolist())
    sitk.WriteImage(sitk_img, tmp_path)
    return ants.image_read(tmp_path)


def densify_surface_points(mesh, nodes_override, target_spacing_mm):
    """Add barycentric-sampled points across every triangle face so the point
    density matches `target_spacing_mm`, instead of relying only on mesh vertices.

    Why this exists: the skin-part VTUs are a sparse hex/shell mesh (mean nearest-
    neighbor spacing ~5mm between the 1,958 vertices) while VOXEL_SIZE_MM is 0.75mm
    -- most voxels between vertices are never hit by the point splat, so after the
    Gaussian blur in voxelize_to_ants() the peak intensity is ~0.01 (vs. ~0.24 for
    the fixed target, whose 405k-point surface is already ~0.9mm-spaced -- close to
    voxel resolution, so it doesn't need this). Only used to build the voxel image
    for registration/Dice -- `nodes_skin_scaled` itself (the real mesh nodes that
    Cell 6/7/8 warp) is untouched.
    """
    # extract_surface() on an all-hex UnstructuredGrid keeps the same point array
    # (order/count), so nodes_override (the scaled coords) can be indexed directly.
    # NOTE: pyvista 0.48.4's extract_surface() no longer accepts an `algorithm=`
    # kwarg (that VTK-algorithm-selection param was removed) -- call with defaults.
    surf = mesh.extract_surface().triangulate()
    faces = surf.faces.reshape(-1, 4)[:, 1:4]
    pts = nodes_override
    v0, v1, v2 = pts[faces[:, 0]], pts[faces[:, 1]], pts[faces[:, 2]]
    area = 0.5 * np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)
    n_samp = np.clip(np.ceil(area / (target_spacing_mm ** 2)).astype(int), 1, None)
    total = int(n_samp.sum())
    extra = np.empty((total, 3))
    k = 0
    for i in range(len(faces)):
        n = n_samp[i]
        r1 = np.sqrt(np.random.rand(n)); r2 = np.random.rand(n)
        a, b, c = 1 - r1, r1 * (1 - r2), r1 * r2
        extra[k:k + n] = a[:, None] * v0[i] + b[:, None] * v1[i] + c[:, None] * v2[i]
        k += n
    return np.vstack([pts, extra])


print('Loading FIXED target VTU...')
fixed_mesh, nodes_fixed = load_vtu(FIXED_VTU)

print('\nAuto-scale check (skin vs target)...')
nodes_skin_scaled, scale_factor = auto_rescale(nodes_skin, nodes_fixed)
skin_centroid = nodes_skin.mean(axis=0)   # anchor for un-scaling later — MUST be
                                           # reused in Cell 8/9, not recomputed from
                                           # the full Head_V6.k point cloud.

# Densify the skin point cloud used for VOXELIZATION ONLY (registration/Dice) --
# spacing tied to VOXEL_SIZE_MM so it stays close to 1 sample/voxel along the
# surface without recomputing it if VOXEL_SIZE_MM changes.
print('\nDensifying sparse skin surface for voxelization (~5mm vertex spacing -> '
      f'~{VOXEL_SIZE_MM*2}mm sample spacing)...')
nodes_skin_dense_scaled = densify_surface_points(skin_mesh, nodes_skin_scaled,
                                                  target_spacing_mm=VOXEL_SIZE_MM * 2)
print(f'  Skin points for voxelization: {len(nodes_skin_scaled):,} (mesh nodes) -> '
      f'{len(nodes_skin_dense_scaled):,} (densified for imaging)')

print(f'\nVoxelizing at {VOXEL_SIZE_MM} mm, dilate+fill-solid (was Gaussian-blur-only) ...')
fixed_ants  = voxelize_to_ants(nodes_fixed,
                               voxel_size=VOXEL_SIZE_MM, dilate_iters=2,
                               tmp_path=os.path.join(OUTPUT_DIR, 'fixed_voxelized.nii.gz'))
moving_ants = voxelize_to_ants(nodes_skin_dense_scaled,
                               voxel_size=VOXEL_SIZE_MM, dilate_iters=2,
                               tmp_path=os.path.join(OUTPUT_DIR, 'skin_voxelized.nii.gz'))

print(f'Fixed  ANTs : {fixed_ants.shape}   max intensity: {fixed_ants.numpy().max():.4f}')
print(f'Moving ANTs : {moving_ants.shape}   max intensity: {moving_ants.numpy().max():.4f}')


## Cell 5 — ANTsPy SyNRA Registration (skin vs skin)

Same three internal stages as the whole-mesh notebook (Rigid → Affine → SyN), but
`flow_sigma` is nudged down from the whole-mesh default of 3.0 — a skin-only match
can afford a tighter local fit without risking the *skull/brain* element quality,
since those aren't in this registration at all. The actual downstream risk (folded
elements) only gets checked in Cell 10, against the full model after Cell 9's
warp — not here.

In [5]:
print(f'Running ANTsPy {SYN_TYPE} registration (skin-only)...')
t0 = time.time()

result = ants.registration(
    fixed   = fixed_ants,
    moving  = moving_ants,
    type_of_transform = SYN_TYPE,

    syn_metric     = 'meansquares',
    syn_sampling   = 0.5,
    reg_iterations = (200, 100, 50),

    flow_sigma     = 2.5,   # was 3.0 in the whole-mesh notebook
    total_sigma    = 0,

    aff_metric     = 'meansquares',
    aff_sampling   = 32,
    aff_iterations = (2100, 1200, 200, 0),

    grad_step      = 0.1,
    verbose        = True,
)

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print('\nForward transforms:')
for tf in result['fwdtransforms']:
    print(f'  {tf}')
print('Inverse transforms:')
for tf in result['invtransforms']:
    print(f'  {tf}')

warped_path = os.path.join(OUTPUT_DIR, 'skin_warped_synra.nii.gz')
ants.image_write(result['warpedmovout'], warped_path)
print(f'\nWarped volume -> {warped_path}')


Running ANTsPy SyNRA registration (skin-only)...
antsRegistration --dimensionality 3 -r [0x76f7d14cc7a8,0x76f7d14cc608,1] --metric meansquares[0x76f7d14cc7a8,0x76f7d14cc608,1,32,regular,0.2] --transform Rigid[0.25] --convergence 2100x1200x1200x0 --smoothing-sigmas 3x2x1x0 --shrink-factors 4x2x2x1 -x [NA,NA] --metric meansquares[0x76f7d14cc7a8,0x76f7d14cc608,1,32,regular,0.2] --transform Affine[0.25] --convergence 2100x1200x1200x0 --smoothing-sigmas 3x2x1x0 --shrink-factors 4x2x2x1 -x [NA,NA] --metric meansquares[0x76f7d14cc7a8,0x76f7d14cc608,1,0.5] --transform SyN[0.100000,2.500000,0.000000] --convergence [200x100x50,1e-7,8] --smoothing-sigmas 2x1x0 --shrink-factors 4x2x1 -u 0 -z 1 --output [/tmp/tmpgp7kqwxr,0x76f7d14cc5e8,0x76f7d14cc648] -x [NA,NA] --float 1 --write-composite-transform 0 -v 1
All_Command_lines_OK
Using single precision for computations.
The composite transform comprises the following transforms (in order): 
  1. Center of mass alignment using fixed image: 0x76f7d14cc7

## Cell 5b — Metric 1: Image-Domain Dice / IoU

**What it measures:** overlap between the warped skin volume and the fixed target
volume — i.e. did the skin surface actually converge onto the target shape.
**Why this metric:** it's the direct, literal measure of registration quality for
this stage's own objective (skin → skin correspondence), and it's threshold-free —
Dice/IoU is bounded [0,1] and interpretable regardless of mesh resolution or units.
**Why it is not sufficient alone:** Dice is computed in the *voxelized image domain*.
It says nothing about whether individual solid elements in the full `Head_V6.k`
survive the same warp without folding — that is a separate, and for LS-DYNA more
important, gate checked in Cell 10.

In [ ]:
def norm(a):
    a = a.astype(np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)


def show_synra_result(fixed_ants, warped_ants, title='Skin_SyNRA_Result'):
    f_arr = norm(fixed_ants.numpy())
    m_arr = norm(warped_ants.numpy())
    z, y, x = [d//2 for d in f_arr.shape]
    views = [
        (f_arr[z,:,:], m_arr[z,:,:], 'Axial'),
        (f_arr[:,y,:], m_arr[:,y,:], 'Coronal'),
        (f_arr[:,:,x], m_arr[:,:,x], 'Sagittal'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for col, (fs, ms, label) in enumerate(views):
        rgb = np.zeros((*fs.shape, 3), dtype=np.float32)
        rgb[...,0] = fs; rgb[...,1] = ms
        axes[0,col].imshow(rgb, origin='lower')
        axes[0,col].set_title(f'Blend — {label}  Red=Fixed  Green=Skin')
        axes[0,col].axis('off')
        diff = np.abs(fs - ms)
        im = axes[1,col].imshow(diff, cmap='hot', origin='lower')
        plt.colorbar(im, ax=axes[1,col], fraction=0.046)
        axes[1,col].set_title(f'Difference — {label}  (darker=better)')
        axes[1,col].axis('off')
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f'{title}.png')
    plt.savefig(out, dpi=120); plt.show()
    print(f'Saved -> {out}')


show_synra_result(fixed_ants, result['warpedmovout'])

# Threshold on min-max NORMALIZED intensities, not raw voxel values. The moving
# (skin) volume is a sparse point-splat (~2k points) that, after Gaussian blur,
# peaks around ~0.01 -- an order of magnitude below a raw 0.1 cutoff -- while the
# denser fixed target (405k points) peaks around ~0.24. A raw threshold makes the
# skin binary mask empty (Dice/IoU forced to 0) regardless of registration quality;
# normalizing each volume to its own [0,1] range first (same as the norm() used for
# the plot above) makes 0.1 a consistent 10%-of-peak cutoff for both volumes.
f_bin = (norm(fixed_ants.numpy()) > 0.1).astype(np.float32)
m_bin = (norm(result['warpedmovout'].numpy()) > 0.1).astype(np.float32)
intersection = (f_bin * m_bin).sum()
dice = 2 * intersection / (f_bin.sum() + m_bin.sum() + 1e-8)
iou  = intersection / (np.clip(f_bin + m_bin, 0, 1).sum() + 1e-8)
print(f'\nDice : {dice:.4f}  (1.0 = perfect overlap)')
print(f'IoU  : {iou:.4f}')
DICE_OK = dice >= 0.85
if DICE_OK:        print('Metric 1 PASS -- skin surfaces converged')
elif dice >= 0.70: print('Metric 1 MARGINAL -- consider raising reg_iterations or lowering flow_sigma further')
else:              print('Metric 1 FAIL -- check scale_factor / mesh orientation before proceeding')


## Cell 6 — Verify Point-Transform Direction (correctness gate)

`ants.apply_transforms_to_points()` maps in the **opposite** direction from
`ants.apply_transforms()` on images: `fwdtransforms` warps the moving *image* onto
the fixed image, but applied to *points* it maps fixed-space → moving-space. Cell 5b's
Dice/IoU is an image-domain check and does **not** validate which direction is
correct for points — get this backwards and the skin (and later the whole head)
warps *away* from the target while Cell 5b still reports a good Dice score.

This cell empirically resolves it: warp the skin points both ways and keep whichever
one actually reduces the centroid distance to the fixed target.

In [ ]:
# [SUPERSEDED 2026-07-10 -- see fix cell below] def transform_points(points_xyz, transformlist, whichtoinvert):
#     df = pd.DataFrame(points_xyz, columns=['x', 'y', 'z'])
#     out = ants.apply_transforms_to_points(
#         dim=3, points=df, transformlist=transformlist, whichtoinvert=whichtoinvert,
#     )
#     return out[['x', 'y', 'z']].values.astype(np.float64)
#
#
# centroid_fixed  = nodes_fixed.mean(axis=0)
# dist_before = np.linalg.norm(nodes_skin_scaled.mean(axis=0) - centroid_fixed)
#
# candA = transform_points(nodes_skin_scaled, result['fwdtransforms'],
#                           [False] * len(result['fwdtransforms']))
# dist_A = np.linalg.norm(candA.mean(axis=0) - centroid_fixed)
#
# candB = transform_points(nodes_skin_scaled, result['invtransforms'],
#                           [False] * len(result['invtransforms']))
# dist_B = np.linalg.norm(candB.mean(axis=0) - centroid_fixed)
#
# print(f'Centroid distance to target BEFORE warp        : {dist_before:.2f}')
# print(f'Centroid distance using fwdtransforms as points  : {dist_A:.2f}')
# print(f'Centroid distance using invtransforms as points  : {dist_B:.2f}')
#
# if dist_A < dist_before and dist_A <= dist_B:
#     TRANSFORM_LIST, WHICH_INVERT = result['fwdtransforms'], [False] * len(result['fwdtransforms'])
#     print('\n-> Using fwdtransforms: reduced centroid distance, matches original notebook convention.')
# elif dist_B < dist_before:
#     TRANSFORM_LIST, WHICH_INVERT = result['invtransforms'], [False] * len(result['invtransforms'])
#     print('\n-> Using invtransforms: fwdtransforms drifted AWAY from target, invtransforms is correct here.')
# else:
#     raise RuntimeError(
#         'Neither transform direction reduces centroid distance to the fixed target -- '
#         'registration itself likely failed (check Cell 5b Dice first).'
#     )
#


## Cell 6-fix (2026-07-10) — RAS→LPS point-transform bug**Error found:** after the Cell 4 solid-fill fix, Dice/IoU (Cell 5b) came out 0.8425–0.8427 across two runs (MARGINAL, up from 0.1788), but the new Cell 7b surface-gap check FAILed hard: mean gap 18.95mm vs a 5.0mm threshold (median 17.99, max 53.86). Diagnosis (via `advisor()` + an out-of-notebook replay against the already-computed transform files, no re-registration needed): the mesh node centroid barely moved under the point transform (47.30mm → 46.86–46.94mm) in **every** `fwdtransforms`/`invtransforms` × `whichtoinvert` combination tried (2 threw `Cannot invert ... because it is not a matrix`, confirming warp fields can't take `whichtoinvert=True`), even though the *image*-domain centroid genuinely closed from 32.64mm → 14.44mm and individual skin points moved a real mean 22.5mm (Cell 7's own printed displacement) — i.e. points were moving, just not converging toward the target, while the image clearly did.**Root cause:** `ants.apply_transforms_to_points()` expects points in ITK's **LPS** convention. The skin-part VTUs originate from Slicer3D (RAS convention, see [[reference_obsidian_vault]]), so `nodes_skin_scaled` was silently fed to the ANTs points API in the wrong sign convention on X/Y the entire time -- a classic ITK/Slicer coordinate-space mismatch, not a `whichtoinvert` direction problem as Cell 6's own docstring assumed.**Fix:** flip X/Y sign (RAS→LPS) immediately before `apply_transforms_to_points`, flip back immediately after. Verified on the existing run's transform files (no re-registration): centroid distance 47.30mm → 24.15mm (both fwd/inv candidates converge to the identical value once the coordinate bug is fixed — direction no longer matters at this precision), and full downstream replay gives surface gap 18.95mm → **15.32mm mean** (still **FAILs** the 5.0mm gate, but real, substantial, reproducible improvement).**Not fully resolved -- do not treat as done:** 15.32mm mean surface gap is still 3x the 5.0mm pass threshold. This RAS/LPS bug was real and worth fixing, but it is not the whole story -- something else is still leaving a ~15mm residual surface mismatch even with correct point-space convention. Candidates for next session: (1) the `auto_rescale()` per-axis scale factor may itself still be off (worth re-checking against the corrected point cloud), (2) the registration's SyN stage converged to a MARGINAL (not PASS) Dice, so the underlying image-domain warp itself may be leaving real residual misalignment that a point-transform fix cannot paper over, (3) `flow_sigma`/`reg_iterations` tuning per Cell 5b's own MARGINAL-branch suggestion has not been tried yet. **Do not proceed to Cell 8 (full `Head_V6.k` apply) until Cell 7b passes** -- Cell 8 reuses this same buggy/fixed `transform_points`, so it was equally broken before this fix, and is still not validated to produce a solver-safe result after it.Full writeup: vault note `2d-3d-regis/Services/Skin-Only SyNRA Registration.md`, section "2026-07-10", and memory [[project_skin_synra_scale_investigation]].

In [ ]:
def transform_points(points_xyz, transformlist, whichtoinvert):
    # RAS -> LPS: ants.apply_transforms_to_points() expects ITK/LPS convention, but
    # these mesh nodes come from Slicer3D-exported VTUs, which use RAS. Flipping only
    # X/Y (the RAS<->LPS axes; Z is shared) fixed a real bug -- see the Cell 6-fix
    # markdown above for the empirical diagnosis (centroid dist 47.30 -> 24.15mm).
    pts_lps = points_xyz.copy()
    pts_lps[:, :2] *= -1
    df = pd.DataFrame(pts_lps, columns=['x', 'y', 'z'])
    out = ants.apply_transforms_to_points(
        dim=3, points=df, transformlist=transformlist, whichtoinvert=whichtoinvert,
    )
    result = out[['x', 'y', 'z']].values.astype(np.float64)
    result[:, :2] *= -1  # LPS -> RAS back, so downstream .vtu/.k writers stay in RAS
    return result


centroid_fixed  = nodes_fixed.mean(axis=0)
dist_before = np.linalg.norm(nodes_skin_scaled.mean(axis=0) - centroid_fixed)

candA = transform_points(nodes_skin_scaled, result['fwdtransforms'],
                          [False] * len(result['fwdtransforms']))
dist_A = np.linalg.norm(candA.mean(axis=0) - centroid_fixed)

candB = transform_points(nodes_skin_scaled, result['invtransforms'],
                          [False] * len(result['invtransforms']))
dist_B = np.linalg.norm(candB.mean(axis=0) - centroid_fixed)

print(f'Centroid distance to target BEFORE warp          : {dist_before:.2f}')
print(f'Centroid distance using fwdtransforms as points   : {dist_A:.2f}')
print(f'Centroid distance using invtransforms as points   : {dist_B:.2f}')

if dist_A < dist_before and dist_A <= dist_B:
    TRANSFORM_LIST, WHICH_INVERT = result['fwdtransforms'], [False] * len(result['fwdtransforms'])
    print('\n-> Using fwdtransforms (RAS/LPS-corrected).')
elif dist_B < dist_before:
    TRANSFORM_LIST, WHICH_INVERT = result['invtransforms'], [False] * len(result['invtransforms'])
    print('\n-> Using invtransforms (RAS/LPS-corrected).')
else:
    raise RuntimeError(
        'Neither transform direction reduces centroid distance to the fixed target even '
        'after the RAS/LPS fix -- registration itself likely failed (check Cell 5b Dice first).'
    )


## Cell 7 — Apply Verified Transform to Skin Nodes (visual QA)

Sanity artifact before touching the full head: warp just the skin mesh with the
direction verified in Cell 6, undo the scale factor, and save it for a quick look
in ParaView next to the fixed target.

In [ ]:
pts_warped = transform_points(nodes_skin_scaled, TRANSFORM_LIST, WHICH_INVERT)

# Undo scale factor about the SAME centroid used in Cell 4 (skin_centroid) — not
# recomputed here, per the anchor-consistency requirement carried into Cell 8/9.
# scale_factor is now a per-axis (3,) vector (see Cell 4) -- np.allclose replaces
# the old scalar `!= 1.0` check.
nodes_skin_registered = (pts_warped - skin_centroid) / scale_factor + skin_centroid \
    if not np.allclose(scale_factor, 1.0) else pts_warped

displacements_skin = np.linalg.norm(nodes_skin_registered - nodes_skin, axis=1)
print(f'Skin displacement -- mean: {displacements_skin.mean():.3f}  max: {displacements_skin.max():.3f}')

registered_skin_mesh = skin_mesh.copy(deep=True)
registered_skin_mesh.points = nodes_skin_registered
registered_skin_mesh.point_data['displacement_mm'] = displacements_skin.astype(np.float32)
skin_out_vtu = os.path.join(OUTPUT_DIR, 'head_skin_synra_registered.vtu')
registered_skin_mesh.save(skin_out_vtu, binary=True)
print(f'Saved -> {skin_out_vtu}')


## Cell 7b — Metric 1b: Surface-to-Surface Gap (KDTree)

Dice/IoU (Cell 5b) is computed on solid-filled volumes, which is good for driving the registration but is forgiving of a uniform few-mm surface offset (huge interior overlap dominates the score). This cell checks actual surface fidelity directly: nearest-neighbor distance from every warped skin point to the fixed target's surface point cloud, the same style of check the friend's notebook uses for the skull (`Gap to head target: mean/max`). Run after Cell 7 so `nodes_skin_registered` exists.

In [ ]:
from scipy.spatial import cKDTree

tree_fixed = cKDTree(nodes_fixed)
surf_gap, _ = tree_fixed.query(nodes_skin_registered)
print(f'Surface gap (warped skin -> fixed target): '
      f'mean {surf_gap.mean():.2f}  median {np.median(surf_gap):.2f}  '
      f'max {surf_gap.max():.2f}  p95 {np.percentile(surf_gap, 95):.2f}  mm')

SURFACE_GAP_OK = surf_gap.mean() < 5.0  # ~scalp-thickness tolerance, same rule of
                                        # thumb as the friend's skull gap check
print('Metric 1b PASS' if SURFACE_GAP_OK else 'Metric 1b FAIL',
      '-- mean surface gap', f'{surf_gap.mean():.2f}mm',
      '(threshold 5.0mm)')


## Cell 8 — Parse Full `Head_V6.k` + Apply the Same Transform to Every Node

Ported from `CT_MRI_Registration_K_Mesh.ipynb` (its Cell 3 parser). Applying
directly to `.k` node coordinates means no VTU conversion is needed on this side
either — `apply_transforms_to_points` only needs an (N,3) array.

Reuses `TRANSFORM_LIST` / `WHICH_INVERT` from Cell 6 and `scale_factor` /
`skin_centroid` from Cell 4 — **not** recomputed from `Head_V6.k`'s own point cloud,
which would shift skull/brain relative to skin and break inter-tissue alignment.

In [ ]:
def parse_k_file(k_path):
    """Parse an LS-DYNA keyword (.k) file's *NODE block, keeping raw lines for a
    round-trip write. Ported unchanged from CT_MRI_Registration_K_Mesh.ipynb."""
    with open(k_path, 'r', errors='replace') as fh:
        raw_lines = fh.readlines()

    nodes, node_ids = [], []
    node_id_map, node_line_indices = {}, {}
    in_node_block = False

    for line_idx, line in enumerate(raw_lines):
        stripped = line.strip()
        if stripped.startswith('*'):
            in_node_block = stripped.upper().startswith('*NODE')
            continue
        if stripped.startswith('$') or stripped == '':
            continue
        if in_node_block:
            try:
                if ',' in stripped:
                    parts = stripped.split(',')
                    nid = int(parts[0].strip())
                    x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                else:
                    nid = int(line[0:8])
                    x   = float(line[8:24])
                    y   = float(line[24:40])
                    z   = float(line[40:56])
                row = len(nodes)
                nodes.append([x, y, z])
                node_ids.append(nid)
                node_id_map[nid] = row
                node_line_indices[nid] = line_idx
            except (ValueError, IndexError):
                continue

    nodes_arr    = np.array(nodes, dtype=np.float64)
    node_ids_arr = np.array(node_ids, dtype=np.int64)
    print(f'Parsed {len(nodes_arr):,} nodes from {k_path}')
    return nodes_arr, node_ids_arr, node_id_map, raw_lines, node_line_indices


print('Parsing full Head_V6.k (every tissue layer)...')
nodes_full, node_ids_full, node_id_map_full, k_raw_lines, node_line_indices = parse_k_file(HEAD_V6_K)
print(f'Full head node count: {len(nodes_full):,}  (skin-only pass used {len(nodes_skin):,} nodes to compute the warp)')

# Same similarity transform anchor as the skin pass (Cell 4) -- reused, not recomputed.
# scale_factor is now a per-axis (3,) vector (see Cell 4) -- np.allclose replaces
# the old scalar `!= 1.0` check.
nodes_full_scaled = (nodes_full - skin_centroid) * scale_factor + skin_centroid \
    if not np.allclose(scale_factor, 1.0) else nodes_full

print(f'\nApplying verified transform to {len(nodes_full_scaled):,} full-head nodes...')
t0 = time.time()
pts_full_warped = transform_points(nodes_full_scaled, TRANSFORM_LIST, WHICH_INVERT)
print(f'Done in {time.time()-t0:.2f}s')

nodes_full_registered = (pts_full_warped - skin_centroid) / scale_factor + skin_centroid \
    if not np.allclose(scale_factor, 1.0) else pts_full_warped

displacements_full = np.linalg.norm(nodes_full_registered - nodes_full, axis=1)
print(f'\nFull-head displacement -- mean: {displacements_full.mean():.3f}  '
      f'max: {displacements_full.max():.3f}  std: {displacements_full.std():.3f}')


## Cell 9 — Write `Head_V6_registered.k`

Round-trip write ported from `CT_MRI_Registration_K_Mesh.ipynb` (its Cell 9): only
the `*NODE` coordinate columns are replaced in place; every other card (elements,
parts, materials, sections, includes) is preserved verbatim from the original file.

In [ ]:
def write_registered_k(raw_lines, node_line_indices, node_ids, nodes_registered, output_path):
    new_lines = list(raw_lines)
    for row_idx, nid in enumerate(node_ids):
        line_idx = node_line_indices[nid]
        original_line = raw_lines[line_idx]
        x, y, z = nodes_registered[row_idx]
        suffix = original_line[56:].rstrip('\n') if len(original_line) > 56 else ''
        new_line = f'{nid:>8d}{x:>16.9E}{y:>16.9E}{z:>16.9E}{suffix}\n'
        new_lines[line_idx] = new_line
    with open(output_path, 'w') as fh:
        fh.writelines(new_lines)
    print(f'Registered .k file written -> {output_path}')


stem = os.path.splitext(os.path.basename(HEAD_V6_K))[0]
out_k_path = os.path.join(OUTPUT_DIR, f'{stem}_registered.k')

write_registered_k(k_raw_lines, node_line_indices, node_ids_full, nodes_full_registered, out_k_path)

nodes_check, _, _, _, _ = parse_k_file(out_k_path)
print(f'\nSanity check: original {len(nodes_full):,} nodes -> output {len(nodes_check):,} nodes')
assert len(nodes_check) == len(nodes_full), 'Node count mismatch -- check writer!'
print('Node count matches.')


## Cell 10 — Metric 2: `k_mesh_qa.py --compare` (solver-safety gate)

**What it measures:** negative/zero-Jacobian (inverted or degenerate) solid
elements, duplicate nodes, and extreme aspect-ratio elements in the *actual .k mesh
that would be sent to LS-DYNA* — checked per-PID so parts with legitimately
different winding conventions don't produce false positives.

**Why this metric, and why it's checked separately from Dice (Cell 5b):** Dice
measures whether the *skin surface* matched the target. It says nothing about what
happened to the internal skull/brain/meninges hex and tet elements once the same
warp field was applied to them in Cell 8 — a `flow_sigma` tight enough to win on
Dice can still fold or invert internal solid elements it never "saw" during
registration. Zero inverted/degenerate elements is a hard requirement for LS-DYNA
(a guaranteed solver crash otherwise), so this is the binding pass/fail gate for the
pipeline, not Dice. `--compare` diffs the warped output against the pre-warp
baseline so any *newly introduced* failures (vs. pre-existing mesh issues) are
called out explicitly.

In [ ]:
print('Running k_mesh_qa.py --compare against the pre-warp baseline...\n')
proc = subprocess.run(
    [sys.executable, K_MESH_QA_PY, out_k_path, '--compare', HEAD_V6_K],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.stderr:
    print('STDERR:', proc.stderr)

QA_PASS = proc.returncode == 0
print('=' * 60)
print(f'Metric 1 (skin Dice >= 0.85)              : {"PASS" if DICE_OK else "FAIL"}')
print(f'Metric 2 (zero inverted/degenerate, k_mesh_qa) : {"PASS" if QA_PASS else "FAIL"}')
print('=' * 60)
if DICE_OK and QA_PASS:
    print(f'\nBoth gates passed -- {out_k_path} is a candidate FEA-ready replacement for Head_V6.k')
else:
    print('\nAt least one gate failed -- do not hand this .k to the solver.')
    if not DICE_OK:
        print('  -> Dice low: check scale_factor, mesh orientation, or raise reg_iterations.')
    if not QA_PASS:
        print('  -> Inverted/degenerate elements introduced: raise flow_sigma (smoother warp) and re-run from Cell 5.')
